# Compare MAGNET incremental phases (legacy / B / C)

Loads timing JSON from `runnable_code/benchmark_magnet_incremental.py`.

Setup: **5 realizations**, **100-day** forecast window, phases `legacy`, `phase_b`, `phase_c`.

Note: mock ETAS params are very productive; uncapped 100-day runs can reach ~1e5 events.
The timed comparison keeps the 100-day horizon but caps forecast events at **500**.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
TIMING = REPO / "outputs" / "magnet_incremental_phase_timing.json"
assert TIMING.is_file(), f"Missing {TIMING}. Run the benchmark first."

data = json.loads(TIMING.read_text())
print("model:", data["model_dir"])
print(
    f"history={data['history_size']}  forecast_days={data['forecast_days']}  "
    f"max_forecast_events={data['max_forecast_events']}  n_runs={data['n_runs']}"
)

In [ ]:
rows = []
for phase, block in data["phases"].items():
    for run in block["runs"]:
        rows.append(
            {
                "phase": phase,
                "label": block["label"],
                "run": run["run"],
                "seed": run["seed"],
                "elapsed_s": run["elapsed_s"],
                "n_events": run["n_events"],
            }
        )
df = pd.DataFrame(rows)
summary = (
    df.groupby(["phase", "label"], sort=False)
    .agg(
        mean_s=("elapsed_s", "mean"),
        std_s=("elapsed_s", "std"),
        min_s=("elapsed_s", "min"),
        max_s=("elapsed_s", "max"),
        mean_events=("n_events", "mean"),
    )
    .reset_index()
)
summary

In [ ]:
phase_order = [p for p in ("legacy", "phase_b", "phase_c", "phase_a") if p in set(df.phase)]
means = [data["phases"][p]["mean_s"] for p in phase_order]
stds = [data["phases"][p]["std_s"] for p in phase_order]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)

ax = axes[0]
x = range(len(phase_order))
ax.bar(
    x,
    means,
    yerr=stds,
    capsize=4,
    color=["#6a645a", "#0b6e4f", "#b35c1e", "#2f5d8c"][: len(x)],
)
ax.set_xticks(list(x))
ax.set_xticklabels(phase_order, rotation=15)
ax.set_ylabel("wall time (s)")
ax.set_title(f"Mean +/- std over {data['n_runs']} realizations")

ax = axes[1]
for phase in phase_order:
    sub = df[df.phase == phase]
    ax.plot(sub["run"] + 1, sub["elapsed_s"], marker="o", label=phase)
ax.set_xlabel("realization")
ax.set_ylabel("wall time (s)")
ax.set_title("Per-realization timings")
ax.legend(fontsize=8)

fig.suptitle(
    f"MAGNET thinning timing | forecast_days={data['forecast_days']} | "
    f"history={data['history_size']}",
    fontsize=11,
)
plt.show()

if "legacy" in data["phases"] and "phase_c" in data["phases"]:
    leg = data["phases"]["legacy"]["mean_s"]
    c = data["phases"]["phase_c"]["mean_s"]
    b = data["phases"].get("phase_b", {}).get("mean_s")
    if c:
        print(f"legacy / phase_c = {leg / c:.2f}x")
    if b and c:
        print(f"legacy / phase_b = {leg / b:.2f}x")
        print(f"phase_b / phase_c = {b / c:.2f}x")

In [ ]:
df

### Re-run benchmark

```bash
cd /path/to/etas_magnet-incremental
export PYTHONPATH="$PWD:$PWD/runnable_code:/path/to/eq_mag_prediction_clean"
python runnable_code/benchmark_magnet_incremental.py \\
  --phases legacy,phase_b,phase_c --n-runs 5 --forecast-days 100 \\
  --max-forecast-events 500 --skip-parity \\
  --out outputs/magnet_incremental_phase_timing.json
```